In [ ]:
from pathlib import Path
import cblind as cb
import matplotlib.pyplot as plt
import numpy as np

TRH_PATH = Path("../Sim_Data/Varying_TRH")
MA0_PATH = Path("../Sim_Data/Varying_MA0")

OUTPUT_DIR = Path("Plots/NSC_pert/ALP_perturbations/ALP_delta_a")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_MASS_EV = 1e-8
EVOLUTION_KAPPA = 110.0
KAPPA_NL_ANALYTIC = 285.0

plt.rcParams.update({
    "pdf.fonttype": 42,
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath, amssymb}",
    "font.family": "serif",
    "font.size": 16,
    "axes.labelsize": 20,
    "axes.titlesize": 18,
    "legend.fontsize": 14,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
})


def find_noax_file(base_path, T_RH):
    trh_folder = base_path / f"TRH_{T_RH * 1e3:.3f}MeV"
    exact = trh_folder / f"noax_NSC_TRH_{T_RH:.6f}.npz"

    if exact.exists():
        return exact

    candidates = sorted(trh_folder.glob("noax_NSC_TRH_*.npz"))
    if not candidates:
        raise FileNotFoundError(f"No no-axion file found for T_RH={T_RH * 1e3:.3f} MeV in {trh_folder}")

    return candidates[0]


def load_scan_data(base_path):
    base_path = Path(base_path)
    axion_files = sorted(base_path.rglob("AX_TRH_*_Tl_*_MA0_*.npz"))

    if not axion_files:
        raise FileNotFoundError(f"No axion files found under {base_path}")

    records = []
    noax_cache = {}

    for axion_file in axion_files:
        with np.load(axion_file, allow_pickle=True) as data:
            axi_perts = data["axiPerts"].item()
            scale_factors = data["scale_factors"].item()
            inputs = data["inputs"].item()

        T_RH = float(inputs["T_RH"])
        ma0 = float(inputs["MA0"])
        ma0_ev = ma0 * 1e9
        R_osc = float(scale_factors["R_osc"])
        R_l = float(scale_factors["R_l"])

        if T_RH not in noax_cache:
            noax_file = find_noax_file(base_path, T_RH)

            with np.load(noax_file, allow_pickle=True) as data:
                horizon = data["horizon"].item()
                modes = data["modes"].item()

            R_RH = float(modes["R_RH"])
            k_RH = float(modes["k_RH"])

            noax_cache[T_RH] = {
                "x": np.asarray(horizon["x"]),
                "kappas": np.asarray(horizon["kappas"]),
                "Phi_ini": np.asarray(horizon["Phi_ini"]),
                "R_RH": R_RH,
                "k_RH": k_RH,
                "H_RH": k_RH / R_RH,
            }

        noax = noax_cache[T_RH]
        kappa_J = np.sqrt(ma0 / noax["H_RH"]) * (R_l / noax["R_RH"])**0.25
        kappa_osc = float(inputs["k_osc"]) / noax["k_RH"]

        records.append({
            **noax,
            "delta_a": np.asarray(axi_perts["delta_a_s"]),
            "T_RH": T_RH,
            "ma0": ma0,
            "ma0_ev": ma0_ev,
            "T_l": float(inputs["Tl"]),
            "R_osc": R_osc,
            "R_l": R_l,
            "x_osc": R_osc / noax["R_RH"],
            "x_l": R_l / noax["R_RH"],
            "kappa_J": kappa_J,
            "kappa_osc": kappa_osc,
        })

    return records


def get_phi_at_kappa(data, kappa):
    idx = np.argmin(np.abs(data["kappas"] - kappa))
    return float(np.abs(data["Phi_ini"][idx]))


trh_records = load_scan_data(TRH_PATH)
ma0_records = load_scan_data(MA0_PATH)

trh_data = {row["T_RH"]: row for row in sorted(trh_records, key=lambda row: row["T_RH"])}
ma0_data = {row["ma0_ev"]: row for row in sorted(ma0_records, key=lambda row: row["ma0_ev"])}

print("T_RH scan:", [f"{T_RH * 1e3:.0f} MeV" for T_RH in trh_data])
print("Mass scan:", [f"{ma0_ev:.1e} eV" for ma0_ev in ma0_data])


In [ ]:
label_x = r"$x$"
label_kappa = r"$\kappa$"
label_delta_a_evol = rf"$|\delta_a(\kappa={EVOLUTION_KAPPA:g},x)|$"
label_delta_a_spec = r"$|\delta_a(\kappa,x=10)|$"

colors_trh, _ = cb.Colorplots().cblind(len(trh_data))
colors_ma0 = plt.cm.plasma(np.linspace(0.0, 0.80, len(ma0_data)))

phi_p = get_phi_at_kappa(next(iter(trh_data.values())), EVOLUTION_KAPPA)
delta_a_i = 2.0 * phi_p
delta_a_r = 48.0 * phi_p
delta_a_f = 1.0

fig_evol, (ax_evol_trh, ax_evol_ma0) = plt.subplots(1, 2, figsize=(13, 4.5))

for i, (T_RH, data) in enumerate(trh_data.items()):
    idx_kappa = np.argmin(np.abs(data["kappas"] - EVOLUTION_KAPPA))
    color = colors_trh[i]
    ax_evol_trh.loglog(data["x"], np.abs(data["delta_a"][idx_kappa]), color=color, label=rf"$T_{{\rm RH}}={T_RH * 1e3:.0f}$ MeV")
    ax_evol_trh.axvline(data["x_l"], color=color, linestyle="--", alpha=0.8)

ax_evol_trh.plot([], [], color="black", linestyle="--", alpha=0.8, label=r"$x_\Lambda$")

for i, (ma0_ev, data) in enumerate(ma0_data.items()):
    idx_kappa = np.argmin(np.abs(data["kappas"] - EVOLUTION_KAPPA))
    color = colors_ma0[i]
    ax_evol_ma0.loglog(data["x"], np.abs(data["delta_a"][idx_kappa]), color=color, label=rf"$m_{{a,0}}={ma0_ev:.1e}$ eV")
    ax_evol_ma0.axvline(data["x_osc"], color=color, linestyle=":", alpha=0.8)

ax_evol_ma0.plot([], [], color="black", linestyle=":", alpha=0.8, label=r"$x_{\rm osc}$")

for ax in (ax_evol_trh, ax_evol_ma0):
    ax.axhline(delta_a_i, color="black", linestyle=":", alpha=0.7)
    ax.text(1e-4, delta_a_i / 5.0, r"$|\delta_a|=2\Phi_{\rm p}$", color="black", va="bottom")
    ax.axhline(delta_a_r, color="black", linestyle="--", alpha=0.7)
    ax.text(1e-4, delta_a_r * 1.3, r"$|\delta_a|=48\Phi_{\rm p}$", color="black", va="bottom")
    ax.axhline(delta_a_f, color="black", linestyle="-.", alpha=0.7)
    ax.set_xlabel(label_x)
    ax.set_ylabel(label_delta_a_evol)
    ax.set_xlim(1e-5, 1e1)
    ax.set_ylim(1e-6, 1e1)
    ax.set_xticks(10.0**np.arange(-5, 2))
    ax.legend(loc="upper left")

fig_evol.tight_layout()
fig_evol.savefig(OUTPUT_DIR / "Modes_delta_a_axion.pdf", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig_evol)

fig_spec, (ax_spec_trh, ax_spec_ma0) = plt.subplots(1, 2, figsize=(13, 4.5))

for i, (T_RH, data) in enumerate(trh_data.items()):
    idx_x = np.argmin(np.abs(data["x"] - 10.0))
    color = colors_trh[i]
    ax_spec_trh.loglog(data["kappas"], np.abs(data["delta_a"][:, idx_x]), color=color, label=rf"$T_{{\rm RH}}={T_RH * 1e3:.0f}$ MeV")
    ax_spec_trh.axvline(data["kappa_J"], color=color, linestyle="-.", alpha=0.7)

for i, (ma0_ev, data) in enumerate(ma0_data.items()):
    idx_x = np.argmin(np.abs(data["x"] - 10.0))
    color = colors_ma0[i]
    ax_spec_ma0.loglog(data["kappas"], np.abs(data["delta_a"][:, idx_x]), color=color, label=rf"$m_{{a,0}}={ma0_ev:.1e}$ eV")
    ax_spec_ma0.axvline(data["kappa_J"], color=color, linestyle="-.", alpha=0.7)

for ax in (ax_spec_trh, ax_spec_ma0):
    ax.plot([], [], color="black", linestyle="-.", alpha=0.7, label=r"$\kappa_J$")
    ax.set_xlabel(label_kappa)
    ax.set_ylabel(label_delta_a_spec)
    ax.set_xlim(1e-1, 4e2)
    ax.set_ylim(1e-6, 50)
    ax.legend(loc="upper left")

fig_spec.tight_layout()
fig_spec.savefig(OUTPUT_DIR / "Spectrums_delta_a_raw.pdf", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig_spec)


In [ ]:
target_mass = min(ma0_data, key=lambda value: abs(np.log10(value / TARGET_MASS_EV)))
data = ma0_data[target_mass]

x = data["x"]
kappas = data["kappas"]
delta_a = np.abs(data["delta_a"])
x_osc = data["x_osc"]
x_l = data["x_l"]
kappa_J = data["kappa_J"]
kappa_osc = data["kappa_osc"]

phi_p = get_phi_at_kappa(data, EVOLUTION_KAPPA)
delta_i = 2.0 * phi_p
delta_r = 48.0 * phi_p
delta_f = 1.0

label_x = r"$x$"
label_kappa = r"$\kappa$"
label_delta = r"$|\delta_a(\kappa,x)|$"

fig, (ax_evol, ax_spec) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"wspace": 0.25})

k_targets = [100.0, 50.0, kappa_osc, 1.0]
colors_k, _ = cb.Colorplots().cblind(len(k_targets))

for i, kappa in enumerate(k_targets):
    idx_k = np.argmin(np.abs(kappas - kappa))
    label = r"$\kappa=\kappa_{\rm osc}$" if np.isclose(kappa, kappa_osc) else rf"$\kappa={kappa:g}$"
    ax_evol.loglog(x, delta_a[idx_k], color=colors_k[i], linestyle="-", label=label)

ax_evol.axhline(delta_i, color="black", linestyle="--", alpha=0.7)
ax_evol.text(2e-5, delta_i / 5.0, r"$|\delta_a|=2\Phi_{\rm p}$", color="black", va="bottom")
ax_evol.axhline(delta_r, color="black", linestyle="--", alpha=0.7)
ax_evol.text(2e-5, delta_r * 1.3, r"$|\delta_a|=48\Phi_{\rm p}$", color="black", va="bottom")
ax_evol.axhline(delta_f, color="black", linestyle="-.", alpha=0.7)
ax_evol.axvline(x_osc, color="black", linestyle="--", alpha=0.7)
ax_evol.text(x_osc / 2.0, 3e-6, r"$x_{\rm osc}$", color="black", rotation=90, va="bottom", ha="left")
ax_evol.axvline(x_l, color="black", linestyle="-.", alpha=0.7)
ax_evol.text(x_l * 1.2, 3e-6, r"$x_\Lambda$", color="black", rotation=90, va="bottom", ha="left")
ax_evol.set_xlabel(label_x)
ax_evol.set_ylabel(label_delta)
ax_evol.set_xlim(1e-5, 1e1)
ax_evol.set_ylim(1e-6, 1e1)
ax_evol.legend(loc="upper left")

x_targets = [10.0, 1.0, x_osc]
colors_x = plt.cm.plasma(np.linspace(0.2, 0.9, len(x_targets)))

for i, x_target in enumerate(x_targets):
    idx_x = np.argmin(np.abs(x - x_target))
    label = r"$x=x_{\rm osc}$" if np.isclose(x_target, x_osc) else rf"$x={x_target:g}$"
    ax_spec.loglog(kappas, delta_a[:, idx_x], color=colors_x[i], label=label, zorder=3)

ax_spec.axvspan(KAPPA_NL_ANALYTIC, kappas[-1], color="gray", alpha=0.4, edgecolor="none", zorder=1, label=r"$\kappa\geq\kappa_{{\rm nl},\phi}$")
ax_spec.axhline(delta_i, color="black", linestyle="--", alpha=0.7, zorder=2)
ax_spec.text(1.5e-1, delta_i / 5.0, r"$|\delta_a|=2\Phi_{\rm p}$", color="black", va="bottom")
ax_spec.axhline(delta_r, color="black", linestyle="--", alpha=0.7, zorder=2)
ax_spec.text(1.5e-1, delta_r * 1.3, r"$|\delta_a|=48\Phi_{\rm p}$", color="black", va="bottom")
ax_spec.axvline(kappa_J, color="black", linestyle="--", alpha=0.5, zorder=2)
ax_spec.text(kappa_J / 1.6, 3e-6, r"$\kappa_J$", color="black", rotation=90, va="bottom", ha="left")
ax_spec.axvline(kappa_osc, color="black", linestyle="-.", alpha=0.5, zorder=2)
ax_spec.text(kappa_osc * 1.8, 3e-6, r"$\kappa_{\rm osc}$", color="black", rotation=90, va="bottom", ha="right")
ax_spec.set_xlabel(label_kappa)
ax_spec.set_ylabel(label_delta)
ax_spec.set_xlim(1e-1, kappas[-1])
ax_spec.set_ylim(1e-6, 1e1)
ax_spec.legend(loc="upper left")

fig.savefig(OUTPUT_DIR / "ALP_overdensity_1e-8_av.pdf", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)
